# Ablation study: trust-region samples vs. random samples

This notebook tests whether the improvement from retraining the neural surrogate comes from trust-region-localized samples or simply from adding the same number of high-fidelity samples.

The three experiment groups are:

1. the saved trust-region result;
2. original training data + trust-region samples;
3. original training data + the same number of newly generated global random samples.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

import follower_solver
from L2Otraining import TrainingConfig, train
from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.nn_solver import NNSolverExperimentConfig, run_nn_solver_experiment
from stackelberg.randomness import make_rng, set_global_seed

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "NNmodel"
RESULTS_DIR = PROJECT_ROOT / "results"

if not (PROJECT_ROOT / "L2Otraining.py").is_file():
    raise RuntimeError("Run this notebook with the project root as the working directory.")

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

T = 24
SEED = 42
upper_net = create_case33bw_20kv()
scenarios = ["SB", "SB", "BG", "BG", "IP"]
lower_nets = [
    pn.create_cigre_network_lv(),
    pn.create_cigre_network_lv(),
    create_cigre_residential_network(),
    create_cigre_residential_network(),
    pn.create_cigre_network_lv(),
]
mg_buses = [17, 32, 15, 21, 1]
unique_scenarios = list(dict.fromkeys(scenarios))
scenario_networks = {}
for scenario, network in zip(scenarios, lower_nets):
    scenario_networks.setdefault(scenario, network)

# Keep these enabled for the complete automated ablation run. Completed random
# datasets are detected and reused, so rerunning the notebook is safe.
RETRAIN_TR_MODELS = True
RUN_TR_NN_SOLVER = True
GENERATE_NEW_SAMPLES = True
RETRAIN_NEW_MODELS = True
RUN_NEW_NN_SOLVER = True

TRAINING_EPOCHS = 500
NN_STEPS = 5000
NN_LEARNING_RATE = 1e-2
NN_SAVE_EVERY = 200
NEW_DATA_SEEDS = {"SB": 4201, "BG": 4202, "IP": 4203}
NEW_DATA_CHECKPOINT_INTERVAL = 100

## 1. Load the saved trust-region result

The filename supplied for this experiment contains `trust_region` (not `turst_region`). The history CSV gives the iteration-level objective trajectory, while the JSON contains the final price vector and full saved history.

In [2]:
TR_RUN_ID = "20260828-045534"
TR_HISTORY_PATH = RESULTS_DIR / f"trust_region_history_{TR_RUN_ID}.csv"
TR_RESULT_PATH = RESULTS_DIR / f"trust_region_result_{TR_RUN_ID}.json"

for required_path in (TR_HISTORY_PATH, TR_RESULT_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

tr_history = pd.read_csv(TR_HISTORY_PATH).sort_values("k").reset_index(drop=True)
with TR_RESULT_PATH.open("r", encoding="utf-8") as file:
    tr_saved_result = json.load(file)

if tr_history.empty:
    raise ValueError(f"Trust-region history is empty: {TR_HISTORY_PATH}")

tr_last = tr_history.iloc[-1]
tr_x_opt = np.asarray(tr_saved_result["x_opt"], dtype=float).reshape(len(scenarios), T)
tr_summary = pd.Series(
    {
        "run_id": TR_RUN_ID,
        "converged": bool(tr_saved_result["converged"]),
        "iterations": int(tr_saved_result["n_iter"]),
        "final_true_objective": float(tr_last["F_current"]),
        "final_candidate_objective": float(tr_last["F_candidate"]),
        "final_radius": float(tr_last["R"]),
        "accepted_steps": int(tr_history["decision"].astype(str).str.startswith("accept").sum()),
        "rejected_steps": int((tr_history["decision"] == "reject").sum()),
    },
    name="trust_region",
)

display(tr_summary.to_frame("value"))
display(tr_history.tail())
display(pd.DataFrame(tr_x_opt, index=scenarios))

,value
run_id,20260828-045534
converged,True
iterations,43
final_true_objective,-101.29897
final_candidate_objective,-101.265342
final_radius,0.000073
accepted_steps,14
rejected_steps,29


,k,F_current,F_candidate,F_tilde_current,F_tilde_candidate,rho,decision,R,n_residuals,rbf_condition_numbers,max_rbf_condition,geometry_resample_attempts
38,38,-101.29897,-101.212983,-101.298964,-101.291247,-inf,reject,0.000306,14,"[136080.759267719, 152033.0393250161, 25413.04...",178589.546750,0
39,39,-101.29897,-101.236453,-101.298964,-101.298955,-inf,reject,0.000214,14,"[147414.59706066543, 137801.71842968994, 26432...",171133.230086,0
40,40,-101.29897,-101.230113,-101.298964,-101.276129,-inf,reject,0.000150,14,"[146082.44069825005, 139299.0839068988, 25273....",172392.403402,0
41,41,-101.29897,-101.262336,-101.298964,-101.272474,-inf,reject,0.000105,14,"[139193.14750776193, 169739.40240783326, 25830...",180162.583261,0
42,42,-101.29897,-101.265342,-101.298964,-101.250425,-inf,reject,0.000073,14,"[133812.0555472771, 146218.9510779727, 28141.9...",182475.276487,0


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB,1.045048,1.091423,1.110336,1.051721,1.108366,1.024407,1.321957,1.034646,1.000332,1.000921,...,1.063006,1.034797,1.499817,1.499958,1.499987,1.499773,1.499988,0.985109,1.125533,1.031857
SB,1.062976,1.127831,1.466499,1.056383,1.039007,1.344635,1.052466,1.057504,0.994752,1.003916,...,1.087294,1.064135,1.494790,1.486213,1.499984,1.499986,1.487902,1.499974,1.398838,1.045117
BG,1.499983,1.499985,1.499985,1.499985,1.499984,1.499985,1.499986,1.499985,1.499982,1.499987,...,1.499986,1.499983,1.499984,1.499983,1.499984,1.499983,1.499985,1.499986,1.499985,1.499985
BG,1.499983,1.499987,1.499985,1.499986,1.499987,1.499987,1.499986,1.499987,1.499985,1.499985,...,1.499985,1.499985,1.499985,1.499987,1.499987,1.499985,1.499986,1.499986,1.499986,1.499987
IP,1.481462,1.481608,1.487102,1.486122,1.482103,1.455926,1.499032,1.484030,1.473870,1.496045,...,1.496380,1.499111,1.499243,1.488322,1.480456,1.493546,1.498827,1.497995,1.499939,1.482495


In [3]:
def load_aligned_arrays(x_path, y_path, *, expected_width=T):
    """Load and validate one aligned X/Y sample pair."""
    x_data = np.load(x_path)
    y_data = np.load(y_path)
    if x_data.ndim != 2 or y_data.ndim != 2:
        raise ValueError(f"Expected 2-D arrays, got X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape != y_data.shape:
        raise ValueError(f"X/Y shape mismatch: X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape[1] != expected_width:
        raise ValueError(f"Expected width {expected_width}, got {x_data.shape[1]}.")
    if not np.isfinite(x_data).all() or not np.isfinite(y_data).all():
        raise ValueError(f"Non-finite values found in {x_path.name} or {y_path.name}.")
    return x_data, y_data


tr_sample_counts = {}
sample_rows = []
for scenario in unique_scenarios:
    original_x, original_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy", DATA_DIR / f"Y_{scenario}.npy"
    )
    tr_x, tr_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_tr.npy", DATA_DIR / f"Y_{scenario}_tr.npy"
    )
    tr_sample_counts[scenario] = len(tr_x)
    sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "trust_region_samples": len(tr_x),
            "combined_samples": len(original_x) + len(tr_x),
        }
    )

sample_count_table = pd.DataFrame(sample_rows).set_index("scenario")
display(sample_count_table)

,original_samples,trust_region_samples,combined_samples
scenario,,,
SB,10000,5190,15190
BG,10000,5176,15176
IP,10000,2595,12595


## 2. Retrain on original + trust-region samples

For each unique scenario, concatenate the original and `_tr` arrays and train a fresh model. `TrainingConfig` retains the original architecture and optimizer settings. Tagged checkpoints ensure that the original models are not overwritten.

In [4]:
tr_training_results = {}
if RETRAIN_TR_MODELS:
    for scenario in unique_scenarios:
        print(f"\nTraining {scenario} on original + trust-region samples")
        tr_training_results[scenario] = train(
            scenario,
            config=TrainingConfig(
                scenario=scenario,
                hidden_dim=128,
                residual_blocks=3,
                batch_size=64,
                learning_rate=1e-3,
                epochs=TRAINING_EPOCHS,
                validation_fraction=0.2,
                seed=SEED,
            ),
            data_tag="tr",
            model_tag="tr",
            include_original=True,
        )
else:
    print("Skipped trust-region model retraining.")

if tr_training_results:
    display(pd.DataFrame(tr_training_results).T.rename_axis("scenario"))


Training SB on original + trust-region samples


SB: concatenated original 10000 + tr 5190 samples -> 15190


SB epoch   10/500: train=0.036368, validation=0.037152


SB epoch   20/500: train=0.026342, validation=0.027092


SB epoch   30/500: train=0.021046, validation=0.021565


...

IP epoch  470/500: train=0.442306, validation=1.133513


IP epoch  480/500: train=0.557917, validation=1.243872


IP epoch  490/500: train=0.430959, validation=1.135264


IP epoch  500/500: train=0.546139, validation=1.240004


,train_mse,validation_mse
scenario,,
SB,0.000917,0.001143
BG,0.000961,0.001077
IP,0.546139,1.240004


In [5]:
tr_nn_result = None
if RUN_TR_NN_SOLVER:
    tr_nn_config = NNSolverExperimentConfig(
        upper_net=upper_net,
        lower_nets=lower_nets,
        mg_buses=mg_buses,
        scenarios=scenarios,
        T=T,
        seed=SEED,
        device="cpu",
        steps=NN_STEPS,
        lr=NN_LEARNING_RATE,
        save_every=NN_SAVE_EVERY,
        history_path=str(RESULTS_DIR / "NN_solver_tr_ablation_history.csv"),
        evaluate_true=True,
        save_results=True,
        model_tag="tr",
    )
    tr_nn_result = run_nn_solver_experiment(tr_nn_config)
    print("TR-retrained NN history:", tr_nn_result["history_path"])
    print("TR-retrained NN true objective:", tr_nn_result["true_obj"])
    print("TR-retrained NN runtime (s):", tr_nn_result["runtime_seconds"])
else:
    print("Skipped the NN solver with trust-region-retrained models.")

Set parameter Username


Set parameter LicenseID to value 2711611


Academic license - for non-commercial use only - expires 2026-09-22


[0] loss=5740.4990, obj=-123.2677, eq=5.8566, soc=0.0647, bound=0.0646, true_loss=1.5415


[200] loss=139.3161, obj=-67.8390, eq=0.2032, soc=0.0367, bound=0.0250, true_loss=-2.2243


...

[4200] loss=-101.4153, obj=-107.9330, eq=0.0064, soc=0.0015, bound=0.0002, true_loss=-88.7156


[4400] loss=-102.2712, obj=-109.3865, eq=0.0070, soc=0.0016, bound=0.0002, true_loss=-90.1581


[4600] loss=-103.5920, obj=-110.0894, eq=0.0063, soc=0.0015, bound=0.0002, true_loss=-91.0252


[4800] loss=-103.9908, obj=-110.7416, eq=0.0066, soc=0.0016, bound=0.0002, true_loss=-91.6942


TR-retrained NN history: /Users/liqi/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Stackelberg Game/Surrogated-assisted Stackelnerg Game/results/NN_solver_tr_ablation_history_20260829-114932.csv
TR-retrained NN true objective: -92.30679830578671
TR-retrained NN runtime (s): 1279.3124418258667


## 3. Retrain on original + equally many global random samples

This control group uses exactly as many new samples as the trust-region group for each scenario, but draws prices globally from $[0, 1.5]^{24}$. The generated files use the `_new` tag and never overwrite the original data.

Generation is checkpointed and resumable. For IP, the date and MIP settings match the production call in `data_generation.py`.

In [6]:
def atomic_save_npy(path, array):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("wb") as file:
        np.save(file, array)
    temporary_path.replace(path)


def atomic_save_json(path, payload):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(payload), encoding="utf-8")
    temporary_path.replace(path)


def generate_tagged_random_samples(
    scenario,
    n_samples,
    network,
    *,
    seed,
    checkpoint_interval=100,
    max_consecutive_failures=20,
):
    """Generate or resume a deterministic `_new` high-fidelity dataset."""
    x_path = DATA_DIR / f"X_{scenario}_new.npy"
    y_path = DATA_DIR / f"Y_{scenario}_new.npy"
    state_path = DATA_DIR / f"generation_{scenario}_new_state.json"

    if x_path.exists() != y_path.exists():
        raise RuntimeError(f"Only one of {x_path.name} and {y_path.name} exists.")

    if x_path.exists():
        saved_x, saved_y = load_aligned_arrays(x_path, y_path)
        completed = len(saved_x)
        if completed > n_samples:
            raise ValueError(
                f"{scenario} already has {completed} new samples; expected at most {n_samples}."
            )
        if completed == n_samples:
            print(f"{scenario}: reusing complete random dataset ({completed} samples).")
            return saved_x, saved_y
        if not state_path.is_file():
            raise RuntimeError(
                f"Cannot safely resume partial {scenario} data without {state_path.name}."
            )
    else:
        saved_x = np.empty((0, T), dtype=float)
        saved_y = np.empty((0, T), dtype=float)
        completed = 0

    set_global_seed(seed)
    rng = make_rng(seed)
    if completed:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if int(state["completed"]) != completed:
            raise RuntimeError(
                f"{scenario} checkpoint state ({state['completed']}) does not match arrays ({completed})."
            )
        rng.bit_generator.state = state["rng_state"]

    solver_kwargs = {"failure_mode": "none", "include_metadata": True}
    if scenario == "IP":
        solver_kwargs.update(date="2023-07-15", time_limit=30.0, mip_gap=1e-3)
    lower_solver = follower_solver.build_solver(
        scenario,
        T=T,
        net=network,
        **solver_kwargs,
    )

    x_data = np.empty((n_samples, T), dtype=float)
    y_data = np.empty((n_samples, T), dtype=float)
    x_data[:completed] = saved_x
    y_data[:completed] = saved_y
    attempts = completed
    failures = 0
    consecutive_failures = 0
    checkpoint_start = time.time()

    print(f"{scenario}: generating samples {completed + 1}..{n_samples} (seed={seed}).")
    while completed < n_samples:
        attempts += 1
        price = rng.uniform(0.0, 1.5, size=T)
        result = lower_solver.lower_solve(price)
        if result is None:
            failures += 1
            consecutive_failures += 1
            print(
                f"{scenario}: skipped failed attempt {attempts}; "
                f"collected {completed}/{n_samples}."
            )
            if consecutive_failures >= max_consecutive_failures:
                raise RuntimeError(
                    f"{scenario}: stopped after {consecutive_failures} consecutive failures."
                )
            continue

        consecutive_failures = 0
        response = np.asarray(result["p_mg"], dtype=float).reshape(T)
        if not np.isfinite(response).all():
            raise ValueError(f"{scenario}: follower returned a non-finite response.")
        x_data[completed] = price
        y_data[completed] = response
        completed += 1

        if completed % checkpoint_interval == 0 or completed == n_samples:
            atomic_save_npy(x_path, x_data[:completed])
            atomic_save_npy(y_path, y_data[:completed])
            atomic_save_json(
                state_path,
                {
                    "scenario": scenario,
                    "seed": seed,
                    "target_samples": n_samples,
                    "completed": completed,
                    "attempts": attempts,
                    "failures": failures,
                    "rng_state": rng.bit_generator.state,
                },
            )
            elapsed = time.time() - checkpoint_start
            print(
                f"{scenario}: saved {completed}/{n_samples} samples "
                f"({failures} failed attempts, {elapsed:.1f} s since last checkpoint)."
            )
            checkpoint_start = time.time()

    return x_data, y_data

In [7]:
new_data = {}
if GENERATE_NEW_SAMPLES:
    for scenario in unique_scenarios:
        new_data[scenario] = generate_tagged_random_samples(
            scenario,
            tr_sample_counts[scenario],
            scenario_networks[scenario],
            seed=NEW_DATA_SEEDS[scenario],
            checkpoint_interval=NEW_DATA_CHECKPOINT_INTERVAL,
        )
else:
    print("Skipped random sample generation; validating existing `_new` arrays.")

new_sample_rows = []
for scenario in unique_scenarios:
    new_x, new_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_new.npy", DATA_DIR / f"Y_{scenario}_new.npy"
    )
    if len(new_x) != tr_sample_counts[scenario]:
        raise ValueError(
            f"{scenario}: expected {tr_sample_counts[scenario]} random samples, got {len(new_x)}."
        )
    original_x = np.load(DATA_DIR / f"X_{scenario}.npy", mmap_mode="r")
    new_sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "new_random_samples": len(new_x),
            "trust_region_samples": tr_sample_counts[scenario],
            "combined_samples": len(original_x) + len(new_x),
        }
    )

new_sample_count_table = pd.DataFrame(new_sample_rows).set_index("scenario")
display(new_sample_count_table)

SB: generating samples 1..5190 (seed=4201).


SB: saved 100/5190 samples (0 failed attempts, 0.2 s since last checkpoint).


SB: saved 200/5190 samples (0 failed attempts, 0.3 s since last checkpoint).


SB: saved 300/5190 samples (0 failed attempts, 0.3 s since last checkpoint).
SB: saved 400/5190 samples (0 failed attempts, 0.2 s since last checkpoint).


SB: saved 500/5190 samples (0 failed attempts, 0.2 s since last checkpoint).
SB: saved 600/5190 samples (0 failed attempts, 0.2 s since last checkpoint).


...

IP: saved 2300/2595 samples (0 failed attempts, 62.8 s since last checkpoint).


IP: saved 2400/2595 samples (0 failed attempts, 53.7 s since last checkpoint).


IP: saved 2500/2595 samples (0 failed attempts, 60.6 s since last checkpoint).


IP: saved 2595/2595 samples (0 failed attempts, 54.4 s since last checkpoint).


,original_samples,new_random_samples,trust_region_samples,combined_samples
scenario,,,,
SB,10000,5190,5190,15190
BG,10000,5176,5176,15176
IP,10000,2595,2595,12595


In [8]:
new_training_results = {}
if RETRAIN_NEW_MODELS:
    for scenario in unique_scenarios:
        print(f"\nTraining {scenario} on original + equally many global random samples")
        new_training_results[scenario] = train(
            scenario,
            config=TrainingConfig(
                scenario=scenario,
                hidden_dim=128,
                residual_blocks=3,
                batch_size=64,
                learning_rate=1e-3,
                epochs=TRAINING_EPOCHS,
                validation_fraction=0.2,
                seed=SEED,
            ),
            data_tag="new",
            model_tag="new",
            include_original=True,
        )
else:
    print("Skipped random-baseline model retraining.")

if new_training_results:
    display(pd.DataFrame(new_training_results).T.rename_axis("scenario"))


Training SB on original + equally many global random samples
SB: concatenated original 10000 + new 5190 samples -> 15190


SB epoch   10/500: train=0.047711, validation=0.049930


SB epoch   20/500: train=0.034699, validation=0.036942


SB epoch   30/500: train=0.022313, validation=0.023949


SB epoch   40/500: train=0.013448, validation=0.014432


...

IP epoch  470/500: train=0.359227, validation=1.152680


IP epoch  480/500: train=0.354144, validation=1.151201


IP epoch  490/500: train=0.345218, validation=1.144249


IP epoch  500/500: train=0.344501, validation=1.143724


,train_mse,validation_mse
scenario,,
SB,0.000417,0.000409
BG,0.000351,0.000406
IP,0.344501,1.143724


In [9]:
new_nn_result = None
if RUN_NEW_NN_SOLVER:
    new_nn_config = NNSolverExperimentConfig(
        upper_net=upper_net,
        lower_nets=lower_nets,
        mg_buses=mg_buses,
        scenarios=scenarios,
        T=T,
        seed=SEED,
        device="cpu",
        steps=NN_STEPS,
        lr=NN_LEARNING_RATE,
        save_every=NN_SAVE_EVERY,
        history_path=str(RESULTS_DIR / "NN_solver_new_ablation_history.csv"),
        evaluate_true=True,
        save_results=True,
        model_tag="new",
    )
    new_nn_result = run_nn_solver_experiment(new_nn_config)
    print("Random-retrained NN history:", new_nn_result["history_path"])
    print("Random-retrained NN true objective:", new_nn_result["true_obj"])
    print("Random-retrained NN runtime (s):", new_nn_result["runtime_seconds"])
else:
    print("Skipped the NN solver with random-retrained models.")

[0] loss=5687.8311, obj=-122.9917, eq=5.8037, soc=0.0647, bound=0.0646, true_loss=2.8869


[200] loss=129.4694, obj=-65.3999, eq=0.1912, soc=0.0345, bound=0.0236, true_loss=-4.8800


[400] loss=29.1090, obj=-52.7444, eq=0.0802, soc=0.0153, bound=0.0104, true_loss=-11.8808


[600] loss=-11.1847, obj=-44.4314, eq=0.0325, soc=0.0070, bound=0.0028, true_loss=-18.0566


[800] loss=-26.4344, obj=-42.1118, eq=0.0153, soc=0.0035, bound=0.0009, true_loss=-23.3599


...

[4200] loss=-98.5043, obj=-105.6713, eq=0.0070, soc=0.0017, bound=0.0001, true_loss=-91.1760


[4400] loss=-99.2709, obj=-106.4981, eq=0.0071, soc=0.0017, bound=0.0001, true_loss=-91.9859


[4600] loss=-100.5687, obj=-107.1851, eq=0.0065, soc=0.0016, bound=0.0001, true_loss=-92.8701


[4800] loss=-100.4491, obj=-107.7764, eq=0.0072, soc=0.0017, bound=0.0001, true_loss=-93.3178


Random-retrained NN history: /Users/liqi/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Stackelberg Game/Surrogated-assisted Stackelnerg Game/results/NN_solver_new_ablation_history_20260829-125323.csv
Random-retrained NN true objective: -93.9988956781133
Random-retrained NN runtime (s): 1274.1698248386383


## 4. Ablation summary

Lower true objective values are better. The key comparison is between the two equally sized augmented datasets: localized trust-region samples versus globally random samples.

In [10]:
def saved_nn_true_objective(history_path):
    history = pd.read_csv(history_path)
    if "true_loss" not in history.columns:
        return np.nan
    evaluated = history.dropna(subset=["true_loss"])
    if evaluated.empty:
        return np.nan
    return float(evaluated.sort_values("iteration").iloc[-1]["true_loss"])


summary_rows = []
original_nn_path = RESULTS_DIR / "NN_solver_history_20260826-145443.csv"
if original_nn_path.is_file():
    summary_rows.append(
        {
            "method": "Original NN surrogate",
            "true_objective": saved_nn_true_objective(original_nn_path),
            "runtime_seconds": np.nan,
            "history_path": str(original_nn_path),
        }
    )

if tr_nn_result is not None:
    summary_rows.append(
        {
            "method": "NN: original + trust-region samples",
            "true_objective": float(tr_nn_result["true_obj"]),
            "runtime_seconds": float(tr_nn_result["runtime_seconds"]),
            "history_path": tr_nn_result["history_path"],
        }
    )

if new_nn_result is not None:
    summary_rows.append(
        {
            "method": "NN: original + equal random samples",
            "true_objective": float(new_nn_result["true_obj"]),
            "runtime_seconds": float(new_nn_result["runtime_seconds"]),
            "history_path": new_nn_result["history_path"],
        }
    )

summary_rows.append(
    {
        "method": "Trust-region residual correction",
        "true_objective": float(tr_summary["final_true_objective"]),
        "runtime_seconds": np.nan,
        "history_path": str(TR_HISTORY_PATH),
    }
)

ablation_summary = pd.DataFrame(summary_rows)
ablation_summary["objective_gap_vs_tr"] = (
    ablation_summary["true_objective"] - float(tr_summary["final_true_objective"])
)
summary_path = RESULTS_DIR / f"retraining_ablation_summary_{time.strftime('%Y%m%d-%H%M%S')}.csv"
ablation_summary.to_csv(summary_path, index=False)

display(ablation_summary)
print("Saved ablation summary to:", summary_path)

,method,true_objective,runtime_seconds,history_path,objective_gap_vs_tr
0,Original NN surrogate,-93.319467,NaN,/Users/liqi/Library/CloudStorage/OneDrive-TheU...,7.979503
1,NN: original + trust-region samples,-92.306798,1279.312442,/Users/liqi/Library/CloudStorage/OneDrive-TheU...,8.992172
2,NN: original + equal random samples,-93.998896,1274.169825,/Users/liqi/Library/CloudStorage/OneDrive-TheU...,7.300075
3,Trust-region residual correction,-101.298970,NaN,/Users/liqi/Library/CloudStorage/OneDrive-TheU...,0.000000


Saved ablation summary to: /Users/liqi/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Stackelberg Game/Surrogated-assisted Stackelnerg Game/results/retraining_ablation_summary_20260829-131437.csv


In [1]:
import gurobipy as gp
from gurobipy import GRB

# max  3x + 4y
# s.t. x + 2y <= 14
#      3x - y >= 0
#      x - y  <= 2
#      x, y   >= 0

model = gp.Model("simple_lp")
x = model.addVar(lb=0.0, name="x")
y = model.addVar(lb=0.0, name="y")

model.setObjective(3 * x + 4 * y, GRB.MAXIMIZE)
model.addConstr(x + 2 * y <= 14, name="c1")
model.addConstr(3 * x - y >= 0, name="c2")
model.addConstr(x - y <= 2, name="c3")

model.optimize()

print(f"Status: {model.Status}")
print(f"x = {x.X:.4f}, y = {y.X:.4f}")
print(f"obj = {model.ObjVal:.4f}")


Set parameter Username
Set parameter LicenseID to value 2711611
Academic license - for non-commercial use only - expires 2026-09-22
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros
Model fingerprint: 0x6228bac5
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 1e+01]
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.2000000e+01   9.996500e+00   0.000000e+00      0s
       1    3.4000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.400000000e+01
Status: 2
x = 6.0000, y = 4.0000
obj = 34.0000
